# 02 - Audit, variables et pretraitement

**Role `tas` (donnees)** - projet *Prediction du succes ou de l'echec des startups*.

Ce notebook montre, sur le dataset Crunchbase reel, les trois etapes dont je suis
responsable dans `src/` :

| Etape | Module | Ce que l'on verifie |
|---|---|---|
| 1. Chargement et audit | `src/data_loader.py` | dimensions, valeurs manquantes, dates aberrantes |
| 2. Variables et cible | `src/features.py`, `src/targets.py` | 6 variables d'entree, aucune fuite de la cible |
| 3. Pretraitement | `src/preprocessing.py` | imputation + encodage + standardisation, appris sur le train seul |

Le decoupage train/test (`src/splitting.py`) appartient a `SA` : il est **appele**
ici pour verifier la chaine complete, jamais modifie.

> Le CSV Kaggle (10,4 Mo) n'est pas versionne. Voir `data/README_DATA.md`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Le notebook vit dans notebooks/ ; les modules dans src/.
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))

import data_loader
import features
import preprocessing
import splitting
import targets

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

print("Variables produites par features.py :", len(features.FEATURE_COLUMNS))
print(features.FEATURE_COLUMNS)
print()
print("Colonnes EDA seulement :", features.EDA_ONLY_COLUMNS)
print("Protocole commun (splitting.INITIAL_FEATURES) :", len(splitting.INITIAL_FEATURES))
print(splitting.INITIAL_FEATURES)

Variables produites par features.py : 6
('country_code', 'state_code', 'region', 'city', 'primary_category', 'founding_year')

Colonnes EDA seulement : ('founding_month',)
Protocole commun (splitting.INITIAL_FEATURES) : 6
('country_code', 'state_code', 'region', 'city', 'primary_category', 'founding_year')


## 1. Chargement et audit du CSV brut

`load_raw` ne modifie rien : il lit le fichier, retire le BOM eventuel et nettoie
les noms de colonnes. L'audit est realise par `audit`, qui ne filtre pas non plus.

In [2]:
chemin = data_loader.find_raw_file(RACINE)
if chemin is None:
    raise FileNotFoundError(
        "CSV introuvable : telecharger le dataset Kaggle dans data/raw/ "
        "(voir data/README_DATA.md)."
    )

print(f"Fichier : {chemin.relative_to(RACINE)}  ({chemin.stat().st_size / 1e6:.1f} Mo)")
brut = data_loader.load_raw(chemin)
rapport = data_loader.audit(brut, current_year=2026)
data_loader.print_report(rapport)

Fichier : data\raw\startup_data.csv  (10.9 Mo)


Lignes : 66368
Colonnes : 14
Doublons complets : 0
Cellules manquantes : 55015

Statuts :
  operating      53034
  closed          6238
  acquired        5549
  ipo             1547

Valeurs manquantes par colonne :
  founded_at                 15221  ( 22.9 %)
  state_code                  8547  ( 12.9 %)
  region                      8030  ( 12.1 %)
  city                        8028  ( 12.1 %)
  country_code                6958  ( 10.5 %)
  homepage_url                5058  (  7.6 %)
  category_list               3148  (  4.7 %)
  first_funding_at              24  (  0.0 %)
  name                           1  (  0.0 %)

Colonne founded_at :
  dates illisibles : 15221
  annee minimale   : 1015
  annee maximale   : 2914
  avant 1900       : 108
  dans le futur    : 4


In [3]:
manquants = pd.DataFrame({
    "colonne": list(rapport["missing"].keys()),
    "valeurs manquantes": list(rapport["missing"].values()),
})
manquants["part (%)"] = (100 * manquants["valeurs manquantes"] / rapport["n_rows"]).round(1)
manquants

,colonne,valeurs manquantes,part (%)
0,founded_at,15221,22.9
1,state_code,8547,12.9
2,region,8030,12.1
3,city,8028,12.1
4,country_code,6958,10.5
5,homepage_url,5058,7.6
6,category_list,3148,4.7
7,first_funding_at,24,0.0
8,name,1,0.0


L'audit confirme ce qui rend ce dataset delicat :

- **22,9 % de `founded_at` manquants** et des dates illisibles ou aberrantes
  (annees avant 1900, dates futures) ;
- des colonnes a forte cardinalite (`city`, `region`, `primary_category`) ;
- un statut `operating` majoritaire, qui ne doit pas etre classe.

Le filtrage des statuts et la construction de la cible sont faits par `SA` dans
`src/targets.py` : `ipo` et `acquired` deviennent 1, `closed` devient 0, et
`operating` est exclu (son issue n'est pas encore observable).

## 2. Variables initiales, cible et decoupage

`build_features` construit les 6 variables du protocole a partir de `category_list`
et `founded_at`. `prepare_target` filtre les statuts. `split_dataset` (SA) decoupe
ensuite en train/test stratifie.

In [4]:
carac = features.build_features(brut, max_year=2026)
print(f"Apres build_features : {carac.shape[0]} lignes, {carac.shape[1]} colonnes")

labellise = targets.prepare_target(carac)
print(f"Apres prepare_target : {labellise.shape[0]} lignes conservees "
      f"({labellise.shape[0] / carac.shape[0]:.1%} du total)")

X_train, X_test, y_train, y_test = splitting.split_dataset(labellise)
print(f"Train : {X_train.shape} | Test : {X_test.shape}")

assert list(X_train.columns) == list(splitting.INITIAL_FEATURES)
assert "founding_month" in carac.columns      # calculee pour l'EDA
assert "founding_month" not in X_train.columns  # jamais une entree du modele
assert not (set(X_train.columns) & splitting.FORBIDDEN_FEATURES)
print("Contrat OK : entrees = protocole SA, aucune colonne interdite.")

Apres build_features : 66368 lignes, 17 colonnes
Apres prepare_target : 13334 lignes conservees (20.1% du total)
Train : (10667, 6) | Test : (2667, 6)
Contrat OK : entrees = protocole SA, aucune colonne interdite.


In [5]:
equilibre = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).round(3)
equilibre.index = ["echec (0)", "succes (1)"]
equilibre

,train,test
echec (0),0.468,0.468
succes (1),0.532,0.532


La stratification est respectee : la part de classe 1 est identique dans le train
et dans le test, donc la comparaison des modeles ne sera pas biaisee par un
desaccord de distribution.

### Pourquoi `founding_month` n'est pas une variable d'entree

`founded_at` n'a souvent que l'annee : Crunchbase code alors le 1er janvier. Le
mois n'est donc pas une vraie saison, mais **une donnee manquante deguisee**.

In [6]:
dates = pd.to_datetime(labellise["founded_at"], errors="coerce", format="mixed")
annee_seule = (dates.dt.month == 1) & (dates.dt.day == 1)
autre_date = dates.notna() & ~annee_seule

artefact = pd.DataFrame({
    "groupe": ["01-01 (annee seule)", "autre date", "date absente"],
    "effectif": [int(annee_seule.sum()), int(autre_date.sum()), int(dates.isna().sum())],
    "part du jeu (%)": [
        round(100 * annee_seule.mean(), 1),
        round(100 * autre_date.mean(), 1),
        round(100 * dates.isna().mean(), 1),
    ],
    "part de classe 1": [
        round(labellise.loc[annee_seule, "target"].mean(), 3),
        round(labellise.loc[autre_date, "target"].mean(), 3),
        round(labellise.loc[dates.isna(), "target"].mean(), 3),
    ],
})
artefact

,groupe,effectif,part du jeu (%),part de classe 1
0,01-01 (annee seule),5676,42.6,0.677
1,autre date,3926,29.4,0.427
2,date absente,3732,28.0,0.422


In [7]:
par_epoque = (annee_seule.groupby(pd.cut(dates.dt.year, [1900, 1999, 2004, 2009, 2012, 2017]))
              .mean().mul(100).round(1))
par_epoque.index = [str(intervalle) for intervalle in par_epoque.index]
par_epoque.to_frame("part de 01-01 (%)")

,part de 01-01 (%)
"(1900, 1999]",89.9
"(1999, 2004]",82.9
"(2004, 2009]",51.3
"(2009, 2012]",40.1
"(2012, 2017]",25.7


Deux constats qui justifient la decision d'equipe :

1. la part de `01-01` **chute avec l'epoque** (dossiers recents mieux renseignes) :
   le mois mesure surtout l'age du dossier ;
2. le taux de classe 1 est tres different entre `01-01` et les autres dates :
   un modele apprendrait un raccourci (anciennete / fenetre d'observation) au lieu
   d'une caracteristique de la startup.

`founding_month` reste donc calculee par `build_features` pour l'analyse
exploratoire (`features.EDA_ONLY_COLUMNS`), mais elle est exclue de
`splitting.INITIAL_FEATURES`. `founding_year` est conservee : elle porte
l'information d'epoque de facon lisible, et sa limite est documentee dans
`docs/methodology.md`.

## 3. Pretraitement commun

Tout traitement qui apprend des parametres (mediane, modalites, moyenne/ecart-type)
est place dans un `Pipeline` ajuste sur `X_train` uniquement. Il est donc re-appris
a chaque fold de validation croisee : aucune information du test ne fuit.

- categorieles : `SimpleImputer("Inconnu")` puis `OneHotEncoder(handle_unknown="ignore")` ;
- numeriques : `SimpleImputer(mediane)` puis `StandardScaler` si `scale=True`.

`scale=True` pour KNN et SVM, `scale=False` pour Zero-R, arbres et Naive Bayes.

In [8]:
resultats = []
for frequence in (None, 10, 20, 50, 100):
    pre = preprocessing.build_preprocessor(scale=True, min_frequency=frequence)
    pre.fit(X_train)
    resultats.append({
        "min_frequency": "aucun" if frequence is None else frequence,
        "colonnes produites": len(preprocessing.feature_names_out(pre)),
    })

pd.DataFrame(resultats)

,min_frequency,colonnes produites
0,aucun,2819
1,10,496
2,20,306
3,50,144
4,100,91


Avec 5 111 villes et 728 secteurs, un encodage complet produit 2 819 colonnes pour
10 667 lignes de train : le sur-apprentissage est garanti. Le regroupement des
modalites rares (`min_frequency=20`, valeur par defaut du module) ramene le jeu a
**306 colonnes** sans perdre les modalites utiles.

In [9]:
pre = preprocessing.build_preprocessor(
    scale=True, min_frequency=preprocessing.DEFAULT_MIN_FREQUENCY
)
X_tr = pre.fit_transform(X_train)
X_te = pre.transform(X_test)

print(f"Train transforme : {X_tr.shape}")
print(f"Test transforme  : {X_te.shape}")
print(f"Valeurs manquantes produites : train={int(np.isnan(X_tr).sum())}, "
      f"test={int(np.isnan(X_te).sum())}")
print(f"Valeurs toutes finies : {bool(np.isfinite(X_tr).all() and np.isfinite(X_te).all())}")

Train transforme : (10667, 306)
Test transforme  : (2667, 306)
Valeurs manquantes produites : train=0, test=0
Valeurs toutes finies : True


In [10]:
noms = preprocessing.feature_names_out(pre)
print(f"{len(noms)} colonnes, dont {sum(n.startswith('cat__') for n in noms)} categorieles "
      f"et {sum(n.startswith('num__') for n in noms)} numerique(s).")
print("Exemples :", noms[:3], "...", noms[-2:])

306 colonnes, dont 305 categorieles et 1 numerique(s).
Exemples : ['cat__country_code_ARG', 'cat__country_code_AUS', 'cat__country_code_BEL'] ... ['cat__primary_category_infrequent_sklearn', 'num__founding_year']


### Preuve de non-fuite

Les parametres appris doivent venir du train seul. On le verifie sur deux points :
la mediane d'imputation et la liste des modalites encodees.

In [11]:
# 1) La mediane est celle du train, pas celle du jeu complet.
imputeur = pre.named_transformers_["num"].named_steps["imputer"]
assert imputeur.statistics_[0] == X_train["founding_year"].median()
print(f"Mediane 'founding_year' apprise : {imputeur.statistics_[0]:.1f} "
      f"== mediane du train ({X_train['founding_year'].median():.1f})")
print(f"Mediane du jeu complet (jamais utilisee) : {labellise['founding_year'].median():.1f}")

# 2) Les modalites encodees sont exactement celles observees dans le train.
sans_regroupement = preprocessing.build_preprocessor(scale=True, min_frequency=None)
sans_regroupement.fit(X_train)
categories = sans_regroupement.named_transformers_["cat"].named_steps["onehot"].categories_
indice_ville = list(preprocessing.CATEGORICAL_FEATURES).index("city")
villes_encodees = set(categories[indice_ville])
villes_train = set(X_train["city"].dropna().unique())
villes_test_seul = sorted(set(X_test["city"].dropna().unique()) - villes_train)

print(f"Villes encodees : {len(villes_encodees)} == villes du train + 'Inconnu' : "
      f"{villes_encodees == villes_train | {preprocessing.MISSING_CATEGORY}}")
print(f"Villes presentes uniquement dans le test : {len(villes_test_seul)}")

Mediane 'founding_year' apprise : 2007.0 == mediane du train (2007.0)
Mediane du jeu complet (jamais utilisee) : 2007.0


Villes encodees : 1515 == villes du train + 'Inconnu' : True
Villes presentes uniquement dans le test : 197


In [12]:
# Une ville ou un secteur jamais vus ne doivent pas faire echouer la transformation.
inconnues = X_test.head(50).copy()
inconnues["city"] = "VilleInexistante"
inconnues["primary_category"] = "SecteurInexistant"
sortie = pre.transform(inconnues)
print(f"Transform de modalites inconnues : {sortie.shape}, manquants={int(np.isnan(sortie).sum())}")

Transform de modalites inconnues : (50, 306), manquants=0


In [13]:
# Pipeline complet, du DataFrame brut a la prediction.
from sklearn.dummy import DummyClassifier

pipeline = preprocessing.build_pipeline(DummyClassifier(strategy="prior"), scale=True)
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
print(f"Zero-R : {len(predictions)} predictions, classes={sorted(set(predictions.tolist()))}")

Zero-R : 2667 predictions, classes=[1]


## Conclusion

| Point | Resultat obtenu |
|---|---|
| Lignes brutes | 66 368 x 14 |
| Lignes conservees apres `prepare_target` | 13 334 (IPO + acquired + closed) |
| Decoupage | 10 667 train / 2 667 test, stratifie (classe 1 = 53,2 % des deux cotes) |
| Variables d'entree | 6 (5 categorieles + `founding_year`) |
| Colonnes apres pretraitement | 2 819 sans regroupement, **306** avec `min_frequency=20` |
| Valeurs manquantes apres pretraitement | 0 |
| Fuite train -> test | aucune : mediane et modalites apprises sur le train seul |

### Choix valides

- `founding_month` exclue des entrees (artefact de precision de date, voir section 2) ;
- `min_frequency=20` : 306 colonnes au lieu de 2 819 ;
- `scale=True` pour KNN et SVM, `scale=False` pour les arbres et Naive Bayes.

### Limites a rappeler dans le rapport

- 22,9 % de dates de creation manquantes : l'imputation par la mediane suppose que
  ces startups ressemblent aux autres, ce qui n'est pas verifiable ;
- l'observabilite de l'issue depend de l'age : une startup ancienne a eu plus de
  temps pour etre acquise, introduite en bourse ou fermer. `founding_year` en
  herite, et aucun fenetrage temporel n'est possible avec ce dataset ;
- les modalites rares sont regroupees : les villes et secteurs peu frequents ne
  sont pas distinguables entre eux.

### Utilisation par les autres roles

```python
from preprocessing import build_pipeline, feature_names_out

modele = build_pipeline(RandomForestClassifier(), scale=False)  # arbres
modele.fit(X_train, y_train)      # imputation + encodage appris ici, train seulement
modele.predict(X_test)

pre = build_preprocessor(scale=True, min_frequency=20)
feature_names_out(pre)            # noms des colonnes pour l'importance des variables
```